# Tweet Sentiment Analysis Using Naive Bayes

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
# Download the list of common stop words
nltk.download('stopwords')

# Download the WordNet database for the lemmatizer
nltk.download('wordnet')
import re

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


## Loading Dataset

In [2]:
data_path = "/content/training.1600000.processed.noemoticon.csv"

In [3]:
import csv
column_names = ['sentiment', 'id', 'date', 'query', 'user', 'text']
df_tweet = pd.read_csv(data_path, encoding='ISO-8859-1', header=None, names=column_names, engine='python', sep=',', quotechar='"', doublequote=True, on_bad_lines='skip')

df_tweet.head()

,sentiment,id,date,query,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [4]:
df_tweet['sentiment'].value_counts()

,count
sentiment,
0,800000
4,800000


In [5]:
from sklearn.utils import resample
n_samples = 120000  # or whatever size you want

df_neg = df_tweet[df_tweet['sentiment'] == 0].sample(n=n_samples, random_state=42)
df_pos = df_tweet[df_tweet['sentiment'] == 4].sample(n=n_samples, random_state=42)  # ← add .sample() here

df_balanced = pd.concat([df_neg, df_pos])
df_balanced['sentiment'].value_counts()

,count
sentiment,
0,120000
4,120000


- Extracting the features and the labels
- Rest of the columns are of no use for us

In [6]:
X = df_balanced['text']
y = df_balanced['sentiment'].replace(4, 1)

In [7]:
print(X.shape)
print(y.shape)

(240000,)
(240000,)


## Data Cleaning

In [8]:
class TweetPreprocessor:
    def __init__(self, normalize='stem'):  # Use 'stem' or 'lemma' instead of 1/2
        """
        normalize: 'stem' for stemming, 'lemma' for lemmatization, None for neither
        """
        self.normalize = normalize

        # Initialize normalizer once (more efficient)
        if normalize == 'stem':
            self.normalizer = PorterStemmer()
        elif normalize == 'lemma':
            self.normalizer = WordNetLemmatizer()
        else:
            self.normalizer = None

    def clean_tweet(self, text):
        # 1. Lowercase
        text = text.lower()
        # 2. Remove URLs (http/https patterns)
        text = re.sub(r'http\S+|www\.\S+', '', text)
        # 3. Handle mentions (@username)
        text = re.sub(r'@\S+', '', text)
        # 4. Handle hashtags (#word)
        text = re.sub(r'#\S+', '', text)
        # 5. Remove special characters
        text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
        # 6. Handle repeated characters (looove -> loove)
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)
        # 7. Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def tokenize(self, text):
        # Split into words
        return text.split()

    def remove_stopwords(self, tokens):
        # Use NLTK stopwords
        stop_words = set(stopwords.words('english'))
        return [word for word in tokens if word not in stop_words]

    def stem_or_lemmatize(self, tokens):
        """Apply stemming or lemmatization based on initialization"""
        if self.normalizer is None:
            return tokens

        if self.normalize == 'stem':
            return [self.normalizer.stem(word) for word in tokens]
        elif self.normalize == 'lemma':
            return [self.normalizer.lemmatize(word) for word in tokens]

        return tokens

    def preprocess(self, text):
        """Complete preprocessing pipeline"""
        text = self.clean_tweet(text)
        tokens = self.tokenize(text)
        tokens = self.remove_stopwords(tokens)
        tokens = self.stem_or_lemmatize(tokens)
        return tokens

# Feature Extraction

## Vocabulary Building

In [9]:
class VocabularyBuilder:
    def __init__(self):
        self.vocab = None
        self.vocab_size = 0
    def build_vocab(self, documents, min_freq=5, max_features=10000):
        """
        documents: list of tokenized tweets
        Returns: word_to_index mapping
        """
        # Count word frequencies across all documents
        word_counts = {}
        for doc in documents:
            for word in doc:
                word_counts[word] = word_counts.get(word, 0) + 1

        # Filter by min_freq
        filtered_words = {word: count for word, count in word_counts.items()
                         if count >= min_freq}

        # Keep top max_features by frequency
        sorted_words = sorted(filtered_words.items(), key=lambda x: x[1], reverse=True)[:max_features]
        # Create word_to_index mapping
        self.vocab = {word: idx for idx, (word, count) in enumerate(sorted_words)}
        self.vocab_size = len(self.vocab)
        return self.vocab

## Feature Representations

### Bag of Words (BoW)

In [10]:
class BagOfWordsVectorizer:
    def __init__(self, vocab):
        """vocab: word_to_index dictionary from VocabularyBuilder"""
        self.vocab = vocab
        self.vocab_size = len(vocab)

    def transform(self, documents):
        """
        documents: list of tokenized tweets
        Returns: (n_samples, vocab_size) matrix of word counts
        """
        # Initialize count matrix
        X = np.zeros((len(documents), self.vocab_size), dtype=np.int16)

        for i, doc in enumerate(documents):
            for word in doc:
                if word in self.vocab:  # Only count words in vocabulary
                    word_idx = self.vocab[word]
                    X[i, word_idx] += 1

        return X

# Naive Bayes Implementation

In [11]:
class MultinomialNaiveBayes:
    def __init__(self, alpha=1.0):
        """
        alpha: Laplace smoothing parameter
        """
        self.alpha = alpha
        self.class_priors = None
        self.feature_probs = None
        self.classes = None

    def fit(self, X, y):
        """
        X: (n_samples, n_features) - word count matrix
        y: (n_samples,) - labels (0 or 1)

        Calculate:
        - P(class) for each class
        - P(word | class) for each word and class
        """
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        # Initialize arrays for priors and likelihoods
        self.class_priors = np.zeros(n_classes)
        self.feature_probs = np.zeros((n_classes, n_features))

        for index, c in enumerate(self.classes):

            # Select only rows belonging to class 'c'
            X_c = X[y == c]

            # Calculate Log Priors: log(count_in_class / total)
            self.class_priors[index] = np.log(X_c.shape[0] / n_samples)

            # Sum word counts per class (Vectorized)
            count_w_c = np.sum(X_c, axis=0)
            total_words_c = np.sum(count_w_c)

            # Log Likelihoods with Laplace Smoothing
            # log((count + 1) / (total + 1 * vocab_size))
            self.feature_probs[index] = np.log((count_w_c + self.alpha) / (total_words_c + self.alpha * n_features))

    def predict(self, X):
        """Fast prediction using log probabilities"""
        log_posteriors = np.dot(X, self.feature_probs.T) + self.class_priors

        return self.classes[np.argmax(log_posteriors, axis=1)]

    def predict_proba(self, X):
        """Return actual probabilities for evaluation/analysis"""
        log_posteriors = X @ self.feature_probs.T + self.class_priors

        # Softmax to convert to probabilities
        log_posteriors -= np.max(log_posteriors, axis=1, keepdims=True)
        probs = np.exp(log_posteriors)
        probs /= np.sum(probs, axis=1, keepdims=True)

        return probs

    def get_top_features(self, vocab, n=20):
        """
        Get most predictive words per class
        vocab: word_to_index dictionary
        n: number of top words to return
        """
        # Reverse vocab to get index_to_word
        index_to_word = {idx: word for word, idx in vocab.items()}

        top_features = {}
        for index, c in enumerate(self.classes):
            # Get indices of words with highest log probability for this class
            top_indices = np.argsort(self.feature_probs[index])[-n:][::-1]

            # Convert indices back to words
            top_words = [(index_to_word[i], self.feature_probs[index][i])
                         for i in top_indices]
            top_features[c] = top_words

        return top_features

# Testing

## Data Split

In [12]:
# Train/Val/Test Split (70/15/15)

# First split: 70% train, 30% temp
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Second split: Split the 30% into 15% validation, 15% test
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train_raw)}, Val: {len(X_val_raw)}, Test: {len(X_test_raw)}")
# Output: Train: 1120000, Val: 240000, Test: 240000

Train: 168000, Val: 36000, Test: 36000


## Preprocess Text

In [13]:
preprocessor = TweetPreprocessor(normalize='stem')

# Preprocess all splits
X_train_tokens = [preprocessor.preprocess(tweet) for tweet in X_train_raw]
X_val_tokens = [preprocessor.preprocess(tweet) for tweet in X_val_raw]
X_test_tokens = [preprocessor.preprocess(tweet) for tweet in X_test_raw]


## Build Vocabulary

In [14]:
vocab_builder = VocabularyBuilder()
vocab = vocab_builder.build_vocab(
    X_train_tokens,
    min_freq=5,      # Word must appear at least 5 times
    max_features=15000  # Keep top 10k words
)


## Vectorize - Convert tokens to numerical features

In [15]:
vectorizer = BagOfWordsVectorizer(vocab)

# Transform all splits using the SAME vocabulary
X_train = vectorizer.transform(X_train_tokens)  # Shape: (1120000, 10000)
X_val = vectorizer.transform(X_val_tokens)      # Shape: (240000, 10000)
X_test = vectorizer.transform(X_test_tokens)    # Shape: (240000, 10000)

## Train Model

In [16]:

model = MultinomialNaiveBayes()
model.fit(X_train, y_train)


## Evaluate on Validation Set (for hyperparameter tuning)

In [ ]:
y_val_pred = model.predict(X_val)
y_val_proba = model.predict_proba(X_val)

from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

val_accuracy = accuracy_score(y_val, y_val_pred)
#val_auc = roc_auc_score(y_val, y_val_proba[])

print(f"Validation Accuracy: {val_accuracy:.4f}")
#print(f"Validation AUC: {val_auc:.4f}")
print(classification_report(y_val, y_val_pred))


# Analyze What Model Learned
#top_features = model.get_top_features(vocab, n=30)

#print("\n=== Top 20 NEGATIVE words ===")
#print(top_features)

Validation Accuracy: 0.7532
              precision    recall  f1-score   support

           0       0.75      0.76      0.76     18000
           1       0.76      0.74      0.75     18000

    accuracy                           0.75     36000
   macro avg       0.75      0.75      0.75     36000
weighted avg       0.75      0.75      0.75     36000


=== Top 20 NEGATIVE words ===
{np.int64(0): [('go', np.float64(-4.279359360611699)), ('get', np.float64(-4.527696256657813)), ('work', np.float64(-4.53299965389543)), ('day', np.float64(-4.714633302216617)), ('miss', np.float64(-4.7907919646694666)), ('like', np.float64(-4.869065341951945)), ('want', np.float64(-4.949235729768656)), ('today', np.float64(-4.983488957840253)), ('feel', np.float64(-5.005320636133861)), ('got', np.float64(-5.142477497129934)), ('realli', np.float64(-5.151435800067457)), ('back', np.float64(-5.1584267985376115)), ('time', np.float64(-5.185988181574117)), ('good', np.float64(-5.21991045627619)), ('im', np.flo

## Evaluation on Test Set

In [18]:

y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_proba[:, 1])

print(f"\n=== FINAL TEST RESULTS ===")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(classification_report(y_test, y_test_pred))


=== FINAL TEST RESULTS ===
Test Accuracy: 0.7530
Test AUC: 0.8255
              precision    recall  f1-score   support

           0       0.75      0.76      0.75     18000
           1       0.75      0.75      0.75     18000

    accuracy                           0.75     36000
   macro avg       0.75      0.75      0.75     36000
weighted avg       0.75      0.75      0.75     36000



# Test on New Tweet

In [19]:
def predict_new_tweet(tweet_text):
    """Predict sentiment of a new tweet"""
    # Preprocess
    tokens = preprocessor.preprocess(tweet_text)
    # Vectorize
    X_new = vectorizer.transform([tokens])
    # Predict
    prediction = model.predict(X_new)[0]
    probabilities = model.predict_proba(X_new)[0]

    sentiment = "Positive" if prediction == 1 else "Negative"
    confidence = probabilities[prediction] * 100

    print(f"Tweet: {tweet_text}")
    print(f"Sentiment: {sentiment} ({confidence:.1f}% confident)")
    print(f"Probabilities: Negative={probabilities[0]:.3f}, Positive={probabilities[1]:.3f}")

    return prediction, probabilities

# Test it!
predict_new_tweet("I absolutely love this product! Best purchase ever!")
predict_new_tweet("This is terrible. Waste of money.")

Tweet: I absolutely love this product! Best purchase ever!
Sentiment: Positive (95.4% confident)
Probabilities: Negative=0.046, Positive=0.954
Tweet: This is terrible. Waste of money.
Sentiment: Negative (94.3% confident)
Probabilities: Negative=0.943, Positive=0.057


(np.int64(0), array([0.94343865, 0.05656135]))